In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.models.mpnn import LigandMPNN

In [ ]:
system = System([
    Protein(
        id="EcCM", rep="TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH", first_index=2
    ),
])
print(f"Sequence length: {len(system[0].rep)}")

# 1. Fold with Boltz, to be replaced with a generated sequence from BoltzGen

In [ ]:
# Alternatively, you can use 
# # retrieve and add evolutionary sequences to system
# system = add_sequences_mmseqs2(
#     system, use_pairing=False, use_env=True
# )

# # perform some extra redundancy reduction on sequences which ColabFold server does not handle with env=True
# system[0].sequences = filter_entity_sequences_mmseqs(
#     system[0], max_seq_id=0.90
# )
# boltz = BoltzFoldTransformer(
#     device='cuda',
#     use_msa=True,
#     diffusion_samples=1,
# )

from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

system = add_sequences_mmseqs2(system, use_pairing=False)

boltz = BoltzFoldTransformer(
    device='cuda',
    use_msa=True,
    diffusion_samples=3,
)

boltz.build(system)
folded = boltz.transform([system.rep_to_instance()])

print(f"Score: {folded[0].score}")
print(f"Confidence (pLDDT): {folded[0].confidence}")

In [ ]:
ei = folded[0][0]
chain_id = list(ei.models.keys())[0]
structure = ei.models[chain_id]
print(f"Chain: {chain_id}")
print(f"Atom count: {len(structure.atom_array)}")

In [ ]:
!pip install py3Dmol -q
import io
import py3Dmol

buf = io.StringIO()
structure.to_file(buf, format="cif")

view = py3Dmol.view(width=800, height=400)
view.addModel(buf.getvalue(), "cif")
view.setStyle({
    "cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "roygb",
            "min": 50,
            "max": 90,
        }
    }
})
view.zoomTo()
view.show()

In [ ]:
s_designable = system.apply_instance(folded[0])
print(f"Structures attached: {s_designable[0].structures is not None}")
print(f"Structure keys: {list(s_designable[0].structures.keys())}")

# 2. Run MPNN sequence optimisation

In [ ]:
mpnn = LigandMPNN(model_name='proteinmpnn_v_48_020')
mpnn.build(s_designable)
designs = mpnn.generate(num_designs=5)

print(f"Generated {len(designs)} designs")
for i, d in enumerate(designs):
    seq_designed = "".join(d[0].rep)
    print(f"  Design {i}: score={d.score:.3f} seq={seq_designed[:30]}...")

In [ ]:
designs

# 3. Refold with Boltz2

## Step 4: Refold the designs with Boltz (single-sequence mode)

To validate the MPNN-designed sequences we refold them with Boltz in single-sequence mode (`use_msa=False`), the usual workflow for checking designed sequences. Each design differs from the wild type by 70-80% of residues, so it has few or no natural homologs: an MSA adds little signal, and reusing the WT MSA would mis-condition Boltz (its hits are column-aligned to the WT, not to the designs).

All designs are instances of the same system, so we fold them together in a single `transform()` call.

In [ ]:
# Validate the MPNN designs by refolding them with Boltz in single-sequence
# mode (use_msa=False) — no MSA is fetched. All designs are instances of the
# same system, so they are folded together in a single transform() call.
boltz_validate = BoltzFoldTransformer(
    device='cuda',
    use_msa=False,
    sampling_steps=200,
    diffusion_samples=1,
).build(s_designable)

refolded = boltz_validate.transform(designs)

print(f"Refolded {len(refolded)} designs (single-sequence mode)")

In [ ]:
refolded

In [ ]:
best = max(refolded, key=lambda r: r.score)
best_idx = refolded.index(best)
print(f"Best design: {best_idx} (score={best.score:.4f})")

ei_best = best[0]
chain_id = list(ei_best.models.keys())[0]
structure_best = ei_best.models[chain_id]

buf = io.StringIO()
structure_best.to_file(buf, format="cif")

view = py3Dmol.view(width=800, height=400)
view.addModel(buf.getvalue(), "cif")
view.setStyle({
    "cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "roygb",
            "min": 50,
            "max": 90,
        }
    }
})
view.zoomTo()
view.show()

# 4. Compare to original structure

In [ ]:
import pandas as pd

seq_original = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
seq_best = "".join(best[0].rep)

matches = sum(a == b for a, b in zip(seq_original, seq_best))
identity = matches / len(seq_original) * 100

print(f"Sequence identity to original: {identity:.1f}%")
print(f"Original score:    {folded[0].score:.4f}")
print(f"Best design score: {best.score:.4f}")
print(f"Improvement:       {best.score - folded[0].score:.4f}")

In [ ]:
import biotite.structure as struc
import numpy as np

def compute_rmsd(reference_structure, mobile_structure):
    """
    Compute CA-only RMSD between two structures after
    superimposition. Uses only backbone CA atoms for
    a fair comparison of overall fold similarity.
    """
    ref_ca = reference_structure.atom_array[
        reference_structure.atom_array.atom_name == "CA"
    ]
    mob_ca = mobile_structure.atom_array[
        mobile_structure.atom_array.atom_name == "CA"
    ]

    min_len = min(len(ref_ca), len(mob_ca))
    if min_len == 0:
        return None
    ref_ca = ref_ca[:min_len]
    mob_ca = mob_ca[:min_len]

    fitted, _ = struc.superimpose(ref_ca, mob_ca)
    return struc.rmsd(ref_ca, fitted)

mpnn_scores = [d.score for d in designs]

original_structure = folded[0][0].models[
    list(folded[0][0].models.keys())[0]
]

print("RMSD to original fold (CA atoms, after superimposition):")
print(f"{'Design':>8} {'RMSD (Å)':>10} {'Boltz score':>12} {'MPNN score':>12}")
print("-" * 46)

rmsd_values = []
for i, r in enumerate(refolded):
    ei = r[0]
    if ei.models is None:
        print(f"{i:>8} {'failed':>10}")
        rmsd_values.append(None)
        continue
    chain_id = list(ei.models.keys())[0]
    design_structure = ei.models[chain_id]
    rmsd_val = compute_rmsd(original_structure, design_structure)
    rmsd_values.append(rmsd_val)
    print(
        f"{i:>8} {rmsd_val:>10.2f} "
        f"{r.score:>12.4f} "
        f"{mpnn_scores[i]:>12.3f}"
    )

valid = [v for v in rmsd_values if v is not None]
best_rmsd_idx = int(np.argmin(valid))
print(f"\nMost similar to original fold: design {best_rmsd_idx} "
      f"(RMSD={rmsd_values[best_rmsd_idx]:.2f} Å)")
print(f"Best Boltz score:  design {refolded.index(best)} "
      f"(score={best.score:.4f})")


In [ ]:
import io
from pathlib import Path
from datetime import datetime

# Create timestamped output directory next to the notebook
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

# Save original fold
original_chain = list(folded[0][0].models.keys())[0]
original_structure = folded[0][0].models[original_chain]
original_path = output_dir / "original_fold.cif"
original_structure.to_file(str(original_path), format="cif")
print(f"Saved original fold: {original_path}")

# Save all refolded designs
design_paths = []
for i, r in enumerate(refolded):
    ei = r[0]
    if ei.models is None:
        print(f"Design {i}: no structure, skipping")
        design_paths.append(None)
        continue
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]
    path = output_dir / f"design_{i}_boltz{r.score:.4f}_mpnn{mpnn_scores[i]:.3f}.cif"
    structure.to_file(str(path), format="cif")
    design_paths.append(path)
    print(f"Saved design {i}: {path.name}")

print(f"\nAll files saved to: {output_dir.resolve()}")
print(f"\nTo align in PyMOL, run:")
print(f"  reinitialize")
print(f"  load {original_path.resolve()}, original_fold")
for i, dpath in enumerate(design_paths):
    if dpath is None:
        continue
    print(f"  load {dpath.resolve()}, design_{i}")
for i, dpath in enumerate(design_paths):
    if dpath is None:
        continue
    print(f"  align design_{i}, original_fold")
